In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs
from databricks.sdk.service.jobs import PauseStatus, CronSchedule, TriggerSettings, TriggerType
from pyspark.sql.functions import col, from_unixtime
from pyspark.sql.types import IntegerType, LongType, DecimalType, StringType, TimestampType

In [0]:
_ambiente = "PRD"

if 'prd' in _ambiente.lower():
    ambiente = _ambiente.lower()
elif 'qa' in _ambiente.lower():
    ambiente = _ambiente.lower()
else:
    ambiente = 'dev'

print("AMBIENTE: ",ambiente)

## Variables for workflows

In [0]:
# name of workflows that will not be deactivated
dbutils.widgets.text("ListJobsNotPaused", "pipe_desativa_workflows")

In [0]:
JobsNotDisable = dbutils.widgets.get("ListJobsNotPaused")
print("workflows that will not be deactivated: ", JobsNotDisable)

In [0]:
# function to disable workflow triggers
def disable_schedule(job_id, trigger_type):
    print(f"Disabling  Job {job_id} with {trigger_type} trigger")
    # variable with the json settings of the desired job
    job_settings = w.jobs.get(job_id=job_id).settings

    if trigger_type == "file_arrival" or trigger_type == "periodic":
        updated_schedule = TriggerSettings(
            pause_status=PauseStatus.PAUSED
        )
        job_settings.trigger = updated_schedule

    elif trigger_type == "schedule":
        updated_schedule = CronSchedule(
            quartz_cron_expression=job_settings.schedule.quartz_cron_expression,
            timezone_id=job_settings.schedule.timezone_id,
            pause_status=PauseStatus.PAUSED  
        )
        job_settings.schedule = updated_schedule

    elif trigger_type == "continuous":
        updated_schedule = TriggerSettings(
            pause_status=PauseStatus.PAUSED
        )
        job_settings.continuous = updated_schedule


    # perform the update on Databricks workflows
    w.jobs.update(job_id=job_id, new_settings=job_settings)

    print(f"Job {job_id} disabled with {trigger_type} trigger")

In [0]:
def get_jobs_workflows():
    # get the workflows
    all_jobs = w.jobs.list()
    
    # Process job data
    job_data = [
        {
            "job_id": job.job_id,
            "name": job.settings.name,
            "creator_user_name": job.creator_user_name,
            "created_time": job.created_time,
            "trigger_type": "schedule" if job.settings.schedule else 
                            "file_arrival" if job.settings.trigger and job.settings.trigger.file_arrival else 
                            "continuous" if job.settings.continuous else
                            "periodic" if job.settings.trigger and not job.settings.trigger.file_arrival
                            else None,
            "trigger_details": job.settings.schedule.quartz_cron_expression if job.settings.schedule else
                            job.settings.trigger.file_arrival.url if job.settings.trigger and job.settings.trigger.file_arrival else
                            "continuous" if job.settings.continuous else None,
            "timezone": job.settings.schedule.timezone_id if job.settings.schedule else None,
            "pause_status": job.settings.schedule.pause_status.value if job.settings.schedule else 
                            job.settings.trigger.pause_status.value if job.settings.trigger else 
                            job.settings.continuous.pause_status.value if job.settings.continuous else None,
            "format": job.settings.format.value,
            "max_concurrent_runs": job.settings.max_concurrent_runs,
            "timeout_seconds": job.settings.timeout_seconds,
            "webhook_alert": bool(job.settings.webhook_notifications.on_failure) if job.settings.webhook_notifications and job.settings.webhook_notifications.on_failure else False
        }
        for job in all_jobs
    ]

    # Create DataFrame in Spark
    jobs_df = spark.createDataFrame(job_data)

    # converts the created time column to date and time
    jobs_df = jobs_df.withColumn("created_time", from_unixtime(col("created_time") / 1000))
    # converts to timestamp
    jobs_df = jobs_df.withColumn("created_time", col("created_time").cast(TimestampType()))
    
    # saves the dataframe to the table in the catalog
    jobs_df.write.mode("overwrite").saveAsTable(f"YOUR_CATALOG.default.workflows")

    print("Tabela de Workflows atualizada")

# Authenticate using the APP

In [0]:
secret_dtb = dbutils.secrets.get(scope=f"az-keyvault", key=f"adb-secret")
tenant_dtb = dbutils.secrets.get(scope=f"az-keyvault", key=f"adb-tenant")
clientId_dtb = dbutils.secrets.get(scope=f"az-keyvault", key=f"adb-application")
host_dtb = dbutils.secrets.get(scope=f"az-keyvault", key=f"adb-host")

w = WorkspaceClient(
    host = host_dtb,
    azure_tenant_id = tenant_dtb,
    azure_client_id = clientId_dtb,
    azure_client_secret = secret_dtb)

### Save the table in the schema `default`

In [0]:
get_jobs_workflows()

### Reads the table to get active workflows with status of `UNPAUSED`

#### Disable triggers if you are in the DEV and QA environment 

** In production maintains status **

In [0]:
if ambiente == 'prd':
    print("Production environment, keeps the workflow trigger status")
else:
    print(f"{ambiente} environment, disabling active workflows")
    
    jobs_df = spark.table(f"YOUR_CATALOG.default.workflows")

    # performs replace in the list passed by the workflow so that it does not deactivate workflows that should not be deactivated
    JobsNotDisable_list = JobsNotDisable.split(",")

    # filters jobs with pause_status like "UNPAUSED"
    unpaused_jobs = (jobs_df
                     .filter((jobs_df.pause_status == "UNPAUSED") 
                             & (~jobs_df.name.isin(JobsNotDisable_list))))

    # selects the job_id and trigger_type columns and collects the results into a list
    unpaused_jobs_list = unpaused_jobs.select("job_id", "trigger_type").collect()

    # list of active workflows
    result_list = [(row.job_id, row.trigger_type) for row in unpaused_jobs_list]

    if result_list:
        # executes the function to disable the workflow trigger
        for job_id, trigger_type in result_list:
            disable_schedule(job_id, trigger_type)
    else: 
        print("No workflows to disable")